<a href="https://colab.research.google.com/github/AmaraRao19/support-ticket-auto-tagging/blob/main/AutoTaggingSupportTickets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Install Required Libraries

In [1]:
!pip install transformers datasets torch scikit-learn

#Create or Load Dataset

In [2]:
import pandas as pd

data = {
    "text": [
        "I can't login to my account",
        "Payment failed but money deducted",
        "App is crashing on startup",
        "I forgot my password",
        "Unable to update profile",
        "Order not delivered yet",
        "Refund not received",
        "Website is too slow",
        "Error while uploading file",
        "Account got hacked"
    ],
    "label": [
        "Login Issue",
        "Payment Issue",
        "Bug",
        "Login Issue",
        "Profile Issue",
        "Delivery Issue",
        "Refund Issue",
        "Performance Issue",
        "Bug",
        "Security Issue"
    ]
}

df = pd.DataFrame(data)
df

,text,label
0,I can't login to my account,Login Issue
1,Payment failed but money deducted,Payment Issue
2,App is crashing on startup,Bug
3,I forgot my password,Login Issue
4,Unable to update profile,Profile Issue
5,Order not delivered yet,Delivery Issue
6,Refund not received,Refund Issue
7,Website is too slow,Performance Issue
8,Error while uploading file,Bug
9,Account got hacked,Security Issue


#Define Possible Tags

In [3]:
labels = list(df['label'].unique())
labels

['Login Issue',
 'Payment Issue',
 'Bug',
 'Profile Issue',
 'Delivery Issue',
 'Refund Issue',
 'Performance Issue',
 'Security Issue']

#Zero-Shot Classification using LLM

In [4]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

#Test Zero-Shot Prediction

In [7]:
text = "My payment was deducted but order not placed"

result = classifier(text, labels, multi_label=True)

print("result:")
result

result:


{'sequence': 'My payment was deducted but order not placed',
 'labels': ['Delivery Issue',
  'Payment Issue',
  'Refund Issue',
  'Security Issue',
  'Profile Issue',
  'Performance Issue',
  'Bug',
  'Login Issue'],
 'scores': [0.9649190902709961,
  0.9522677063941956,
  0.625981867313385,
  0.6010469794273376,
  0.4969300627708435,
  0.3566759526729584,
  0.20361021161079407,
  0.1435333490371704]}

#Get Top 3 Tags

In [8]:
def get_top_3(result):
    labels_scores = list(zip(result['labels'], result['scores']))
    sorted_labels = sorted(labels_scores, key=lambda x: x[1], reverse=True)
    return sorted_labels[:3]

get_top_3(result)

[('Delivery Issue', 0.9649190902709961),
 ('Payment Issue', 0.9522677063941956),
 ('Refund Issue', 0.625981867313385)]

#Apply to Full Dataset

In [9]:
df['predictions'] = df['text'].apply(lambda x: get_top_3(classifier(x, labels, multi_label=True)))

df

,text,label,predictions
0,I can't login to my account,Login Issue,"[(Login Issue, 0.9938158988952637), (Security ..."
1,Payment failed but money deducted,Payment Issue,"[(Payment Issue, 0.993032693862915), (Delivery..."
2,App is crashing on startup,Bug,"[(Performance Issue, 0.9807319641113281), (Bug..."
3,I forgot my password,Login Issue,"[(Security Issue, 0.9930335879325867), (Login ..."
4,Unable to update profile,Profile Issue,"[(Profile Issue, 0.9920870065689087), (Bug, 0...."
5,Order not delivered yet,Delivery Issue,"[(Delivery Issue, 0.994367778301239), (Payment..."
6,Refund not received,Refund Issue,"[(Refund Issue, 0.9929617047309875), (Payment ..."
7,Website is too slow,Performance Issue,"[(Performance Issue, 0.9663904309272766), (Log..."
8,Error while uploading file,Bug,"[(Bug, 0.9795052409172058), (Performance Issue..."
9,Account got hacked,Security Issue,"[(Security Issue, 0.9923350811004639), (Bug, 0..."


#Few-Shot Learning (Prompt Engineering)

In [12]:
def few_shot_prompt(text):
    prompt = f"""
    Classify the support ticket into categories.

    Examples:
    Text: I can't login
    Tags: Login Issue

    Text: Payment failed
    Tags: Payment Issue

    Text: {text}
    Tags:
    """
    return prompt

Test:

In [13]:
classifier(few_shot_prompt("App crashes when I open it"), labels)

{'sequence': "\n    Classify the support ticket into categories.\n\n    Examples:\n    Text: I can't login\n    Tags: Login Issue\n\n    Text: Payment failed\n    Tags: Payment Issue\n\n    Text: App crashes when I open it\n    Tags:\n    ",
 'labels': ['Login Issue',
  'Payment Issue',
  'Bug',
  'Security Issue',
  'Performance Issue',
  'Profile Issue',
  'Delivery Issue',
  'Refund Issue'],
 'scores': [0.844735860824585,
  0.10675210505723953,
  0.030698103830218315,
  0.004848987329751253,
  0.004651790950447321,
  0.004115833900868893,
  0.0022504141088575125,
  0.0019469248363748193]}

#Compare Results

In [14]:
zero_shot = classifier("App crashes frequently", labels)
few_shot = classifier(few_shot_prompt("App crashes frequently"), labels)

print("Zero-shot:", get_top_3(zero_shot))
print("Few-shot:", get_top_3(few_shot))

Zero-shot: [('Performance Issue', 0.5116540789604187), ('Bug', 0.2455562949180603), ('Profile Issue', 0.07075708359479904)]
Few-shot: [('Login Issue', 0.8602973818778992), ('Payment Issue', 0.09369863569736481), ('Bug', 0.028001854196190834)]


#Save Notebook

In [15]:
df.to_csv("predictions.csv", index=False)